<div align="center">
  <img src="https://upload.wikimedia.org/wikipedia/commons/d/d0/Google_Colaboratory_SVG_Logo.svg" width="50" alt="Colab Logo">
  <p style="margin-bottom: 0px;">Developed by <b>Lakshan</b></p>
  <h1 style="margin-top: 5px;"><b>Archive Vault Cracker 🔐</b></h1>
  <p>High-Speed Cloud Password Recovery Solution</p>
  <a href="https://github.com/lakshan-bandara/Archive-Vault-Cracker">
    <img src="https://img.shields.io/badge/GitHub-Repository-black?style=for-the-badge&logo=github" alt="GitHub">
  </a>
</div>

---

In [ ]:
#@title **Step 1: System Optimization & Tools Installation**
#@markdown Install required tools (fcrackzip, john). This will take a few seconds.

import os
import subprocess
from IPython.display import clear_output

def install_tools():
    print("🚀 Optimizing system environment...")
    subprocess.run(["apt-get", "update", "-qq"], capture_output=True)
    subprocess.run(["apt-get", "install", "-qq", "-y", "fcrackzip", "john"], capture_output=True)
    with open("test_file.txt", "w") as f:
        f.write("This is a secret test file.")
    clear_output()
    print("✅ Environment Ready! Tools installed.")

install_tools()

In [ ]:
#@title **Bonus: Create a Test Protected ZIP**
#@markdown This creates `/content/test_vault.zip` with password `123456` to verify the cracker works.

import subprocess
print("📦 Creating test protected ZIP...")
subprocess.run(["zip", "-P", "123456", "test_vault.zip", "test_file.txt"])
print("✅ Done! Path: /content/test_vault.zip  |  Password: 123456")

In [ ]:
#@title **Step 2: Start Brute-Force Recovery**
Archive_Path = "" #@param {type:"string"}
Method = "John the Ripper (Universal)" #@param ["fcrackzip (Faster for ZIP)", "John the Ripper (Universal)"]
Wordlist_Selection = "Standard (rockyou.txt)" #@param ["Standard (rockyou.txt)", "Custom URL"]
Custom_Wordlist_URL = "" #@param {type:"string"}

import os
import subprocess

def download_wordlist():
    if Wordlist_Selection == "Standard (rockyou.txt)":
        if not os.path.exists("rockyou.txt"):
            print("📥 Downloading wordlist...")
            subprocess.run(["wget", "-q", "https://github.com/brannondorsey/naive-hashcat/releases/download/data/rockyou.txt"])
        return "rockyou.txt"
    else:
        print("📥 Downloading custom wordlist...")
        subprocess.run(["wget", "-q", Custom_Wordlist_URL, "-O", "custom_list.txt"])
        return "custom_list.txt"

def extract_hash(archive_path):
    """Extract hash from archive using rar2john or zip2john via subprocess."""
    hash_file = "temp_hash.txt"
    ext = archive_path.split('.')[-1].lower()
    
    if ext == 'rar':
        tool = "rar2john"
    else:
        tool = "zip2john"
    
    print(f"⚙️ Running {tool} to extract hash...\n")
    result = subprocess.run(
        [tool, archive_path],
        capture_output=True,
        text=True
    )
    
    hash_output = result.stdout.strip()
    
    if not hash_output:
        print(f"⚠️ {tool} stdout empty, checking stderr...")
        hash_output = result.stderr.strip()
    
    if not hash_output:
        print("❌ Error: Could not extract hash. File may be corrupted or format is unsupported.")
        return None
    
    with open(hash_file, "w") as f:
        f.write(hash_output + "\n")
    
    print(f"✅ Hash extracted successfully.")
    return hash_file

def run_recovery():
    if not Archive_Path:
        print("⚠️ Error: Please provide an Archive Path.") ; return
    if not os.path.exists(Archive_Path):
        print(f"⚠️ Error: File not found at {Archive_Path}") ; return

    wl = download_wordlist()
    print(f"\n🔍 Starting recovery on {Archive_Path} using {Method}...\n")
    
    if "fcrackzip" in Method:
        cmd = ["fcrackzip", "-u", "-D", "-p", wl, Archive_Path]
    else:
        hash_file = extract_hash(Archive_Path)
        if not hash_file: return
        cmd = ["john", f"--wordlist={wl}", hash_file]
    
    print("--- PROCESSING ---\n")
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    
    found = False
    for line in iter(process.stdout.readline, b''):
        decoded = line.decode().strip()
        if decoded:
            print(decoded)
            if any(k in decoded.lower() for k in ["password", "found", "hash cracked"]):
                found = True
    process.wait()
    
    # Show cracked password explicitly
    if "fcrackzip" not in Method and os.path.exists("temp_hash.txt"):
        print("\n--- CRACKED PASSWORDS ---")
        show = subprocess.run(["john", "--show", "temp_hash.txt"], capture_output=True, text=True)
        print(show.stdout if show.stdout.strip() else "No cracked passwords to show.")
    
    if found or ("fcrackzip" not in Method and show.stdout.strip() and "0 password hashes cracked" not in show.stdout):
        print("\n✅ Password recovery completed successfully!")
    else:
        print("\n❌ Password not found in wordlist.")

run_recovery()

---
<div align="center">
  <p><b>Contact Me</b></p>
  <a href="https://wa.me/94768855659">
    <img src="https://upload.wikimedia.org/wikipedia/commons/6/6b/WhatsApp.svg" width="30" alt="WhatsApp Logo" style="vertical-align: middle;">
    <span style="font-size: 1.2em; font-weight: bold; vertical-align: middle; margin-left: 5px;">Lakshan</span>
  </a>
</div>